# Suite TXN — Transactions: mutation expressions over frames, and what they are built from

A transaction is a mutation expression over a stream of frames (docs/STREAMING_QUERIES.md; docs/OBJECT_IDENTITY.md,
Decided, 85 to 97): its terms are reference objects of the `Mutations` schemas, linked by relations, each term's
pattern a `Schemas.OfPattern` built through its link, and a term chosen through a kind union, `Mutations.OfTerm`. A
pattern's `spec` says what must hold of a frame, written in the builders' notation over the frame's symbols and recorded
as mutations. A keyed adjacency takes a mapping in place of the peer shorthand (FRAMEWORK.md, Resolved).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Paths, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by name; crm.Note, counted. */
function crm(store: any): any[] {
  const [Int, Str] = [store.Schemas.OfInt, store.Schemas.OfStr];
  const Knows = store.Schemas.OfRelation((r: any) => r.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")])
    .properties({ since: Int }));
  const Person = store.Schemas.OfObject((o: any) => o.name("crm.Person").ref(true).properties({ name: Str, age: Int }));
  const Holds = store.Schemas.OfRelation((r: any) => (
    r.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({ key: Str })
      .uniques([["book", "key"], ["contact"]])));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies([
    (t: any) => t.name("knows").relation([Knows]).me("knower"),
    (t: any) => t.relation([Knows]).me("known"),
    (t: any) => t.name("book").relation([Holds]).me("contact").owner(true),
  ]));
  const Book = store.Schemas.OfObject((o: any) => o.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
    contacts: (t: any) => t.relation([Holds]).me("book") }));
  const Note = store.Schemas.OfObject((o: any) => o.name("crm.Note").ref(true).properties({ text: Str }));
  const Meets = store.Schemas.OfRelation((r: any) => r.name("crm.Meets").links(["a", "b", "c"].map((n) => (t: any) => t.name(n))));
  const Kind = store.Schemas.OfUnion((u: any) => u.name("crm.Kind").branches([(t: any) => t.name("person").schema([Person]),
    (t: any) => t.name("note").schema([Note])]));
  const Picks = store.Schemas.OfRelation((r: any) => r.name("crm.Picks").links([(t: any) => t.name("owner"),
    (t: any) => t.name("pick").schema([Kind])]).properties({ rank: Int }));
  const Pins = store.Schemas.OfRelation((r: any) => r.name("crm.Pins").links([(t: any) => t.name("note"),
    (t: any) => t.name("book").schema([Book])]).properties({ at: Int }));
  const Trio = store.Schemas.OfRelation((r: any) => r.links(["a", "b", "c"].map((n) => (t: any) => t.name(n)))); // unnamed
  const Odd = store.Schemas.OfObject((o: any) => o.name("crm.Odd").ref(true).adjacencies([(t: any) => t.name("x").me("a")])); // no relation yet
  const Odds = store.Schemas.OfRelation((r: any) => r.name("crm.Odds").links([(t: any) => t.name("note"), (t: any) => t.name("odd").schema([Odd])])
    .properties({ k: Int }));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies({ meets: (t: any) => t.relation([Meets]).me("a"),
    trio: (t: any) => t.relation([Trio]).me("a") }));
  store.Schemas.OfObject(Note, (o: any) => o.adjacencies({ picks: (t: any) => t.relation([Picks]).me("owner"),
    pins: (t: any) => t.relation([Pins]).me("note"), odds: (t: any) => t.relation([Odds]).me("note") }));
  return [Person, Knows, Book, Note];
}

const store: any = new B.Store();
const [Int, Str] = [store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()];
const [Person, Knows, Book, Note] = crm(store);

## TXN-01 · A kind union: a link's schema a union of reference object schemas, each branch a kind built through it

In [ ]:
{
  const Mutations = store.Mutations;
  const OfTerm = Mutations.OfTerm.as_schema();
  const kinds = S._kinds(OfTerm) as Map<string, unknown>;
  assert(same([...kinds.keys()], ["map", "reduce", "partition", "update", "upsert"])
    && [...kinds.values()].every((kind, i) => kind === [Mutations.OfMap, Mutations.OfReduce, Mutations.OfPartition,
      Mutations.OfUpdate, Mutations.OfUpsert][i].as_schema()));
  assert(S._kinds(Person) === null && S._kinds(store.Schemas.OfStr.as_schema()) === null);
  assert(store.names().filter((name: string) => name.startsWith("Mutations.")).every((name: string) => store.registered(name).validate().length === 0));
  const mixed = store.Schemas.OfUnion((u: any) => u.branches([(t: any) => t.name("p").schema([Person]),
    (t: any) => t.name("v").schema([store.Schemas.OfObject((o: any) => o)])]));
  assert(same(store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b").schema([mixed])])).validate(),
    ["link 'b': a kind union's branches are reference object schemas"]));
  // Building through the link: a branch, called, builds an object of its kind; the peer shorthand's visitor and an entry
  // builder each have the branches as members.
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-1").mutations([(mu: any) => mu.upsert((u: any) => (
    u.key("k").then([(w: any) => w.update((d: any) => d)])))]));
  const root = tx.mutations[0].term;
  assert(root.as_schema() === Mutations.OfUpsert.as_schema() && root.key.get() === "k");
  assert(root.then[0].next.as_schema() === Mutations.OfUpdate.as_schema());
  const other = store.Mutations.OfTransaction((t: any) => t.uid("t-2").mutations((a: any) => a.insert((e: any) => e.map((m: any) => m))));
  assert(other.mutations[0].term.as_schema() === Mutations.OfMap.as_schema());
  const kept = store.Mutations.OfTransaction((t: any) => t.uid("t-3").mutations((a: any) => a.insert((e: any) => e.update(root.then[0].next))));
  assert(kept.mutations[0].term === root.then[0].next); // a kind given an object links it
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.uid])), // a member of no kind: which object?
    "link 'term' is a kind union: build through one of its kinds, map, reduce, partition, update, upsert");
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => e.term((x: any) => x)))),
    "link 'term' is a kind union: build through one of its kinds");
  const seen: string[] = [];
  store.Mutations.OfTransaction((t: any) => t.mutations((a: any) => a.insert((e: any) => (seen.push(repr(e.upsert)), e.upsert((u: any) => u)))));
  assert(same(seen, ["<kind of link 'term'>"]));
}

## TXN-02 · Mappings for keyed adjacencies: each item an entry, its key the one property, its value a callback or given to the one member that takes it

In [ ]:
{
  assert(same([...Person.properties.keys()], ["name", "age"]) && Person.properties.get("age").schema === Int); // the schema builders'
  const Tagged = store.Schemas.OfObject((o: any) => o.properties({ tag: (t: any) => t.schema([Str]).description("a label") }));
  assert(Tagged.properties.get("tag").description === "a label");
  raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas({ n: Int })),
    "'schemas' takes a mapping only where its relation has two links and one property");
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: "text" })), // its description and its naming pattern
    "more than one member of the entry takes 'text': give a callback");
  raises(TypeError, () => store.Schemas.OfObject((o: any) => o.properties({ x: 3n })), "no member of the entry takes 3: give a callback");
  const rootModule = store.module();
  store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ Tagged })); // the link alone takes a schema
  assert(rootModule.schemas.get("Tagged") === Tagged);
  const book = store.singleton("crm.TheBook");
  store.crm.Book(book, (b: any) => b.contacts({ ada: (e: any) => e.name("Ada"), bob: (e: any) => e.name("Bob").age(36n) }));
  const [ada, bob] = [book.contacts({ key: "ada" }), book.contacts({ key: "bob" })]; // the proxies': each a contact, by its key
  assert(equal([...book.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ada", "Ada"], ["bob", "Bob"]]) && bob.age.get() === 36n);
  store.crm.Person(ada, (c: any) => c.knows(new Map([[2020n, (e: any) => e.known(bob)]])));
  assert(equal([...ada.knows].map((e: any) => [e.since.get(), e.known]), [[2020n, bob]]));
  raises(TypeError, () => store.crm.Person(ada, (c: any) => c.knows(new Map([[2021n, bob]]))), // the link, and the linked person's own 'knows'
    "more than one member of the entry takes <crm.Person>: give a callback");
  raises(TypeError, () => store.crm.Book(book, (b: any) => b.contacts({ x: 1.5 })), "no member of the entry takes 1.5: give a callback");
  raises(TypeError, () => store.Mutations.OfTransaction((t: any) => t.mutations({ x: (e: any) => e })),
    "adjacency 'mutations' takes a mapping only where its relation has two links and one property");
  raises(TypeError, () => store.Schemas.OfModule(rootModule, (m: any) => m.schemas({ x: "x" })), "no member of the entry takes 'x': give a callback");
  raises(TypeError, () => store.Schemas.OfAny((a: any) => a.schemas("x")),
    "'schemas' takes an iterable of what it links, or of entries, a mapping, or a callback");
  // A link of a kind union, or of a schema it declares, takes an object of it; a callback chooses a kind.
  const note = store.crm.Note((n: any) => n.picks(new Map<bigint, unknown>([[1n, ada], [2n, (e: any) => e.note((m: any) => m.text("aside"))]]))
    .pins(new Map([[3n, book]])));
  assert(equal([...note.picks].map((e: any) => [e.rank.get(), e.pick.as_schema().name]), [[1n, "crm.Person"], [2n, "crm.Note"]]));
  assert(equal([...note.pins].map((e: any) => [e.at.get(), e.book]), [[3n, book]]));
  raises(TypeError, () => store.crm.Note((n: any) => n.odds(new Map([[1n, "zz"]]))), // an adjacency with no relation yet takes nothing
    "no member of the entry takes 'zz': give a callback");
  const outside = new B.OfObject.Builder().properties({ n: (t: any) => t.description("d") }).create(); // outside a store too
  assert(outside.properties.get("n").description === "d");
}

## TXN-03 · A transaction's expression: reference objects linked by relations, each term's pattern a schema object built through its link and held while its term is

In [ ]:
{
  const tx = store.Mutations.OfTransaction((t: any) => t.uid("t-9").mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ ada: Person }).description("whom")]))]));
  const pattern = tx.mutations[0].term.exists[0].pattern;
  assert(pattern instanceof S.OfPattern.Data && same([...pattern.symbols.keys()], ["ada"]) && pattern.description === "whom");
  assert(store.holds(pattern) && store.holds(pattern.symbols.get("ada"))); // weakly: it goes with its transaction (Decided, 94)
  raises(LookupError, () => Paths.of(pattern), "a pattern with no name of its own");
  const Edit = store.Schemas.OfPattern((p: any) => p.name("crm.Edit").symbols({ ada: Person }));
  store.Mutations.OfUpdate((u: any) => u.apply([(p: any) => p.pattern(Edit).description("edits")])); // linked, then updated
  assert(Edit.description === "edits");
  assert(store._meta_builder("crm.Person") === null && store._meta_builder("Schemas.Nope") === null);
  assert(repr(store._meta_builder("Schemas.OfPattern")) === "<builder of a pattern>");
  assert(store._meta_builder_class(Edit) === B.PatternBuilder);
}

## TXN-04 · A spec: written in the builders' notation over the frame's symbols, recorded as mutations, by symbol

In [ ]:
{
  const ref = (name: string): any => ({ ref: { symbol: name } });
  const pathOf = (target: unknown): any => new Map([["ref", new Map([["path", Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(target))]])]]);
  // (a path is a Map already: `map` keeps it)
  const link = (name: string, target: unknown): any => ({ as_link: { name, mutations: [{ set: target }] } });
  const prop = (name: string, value: unknown): any => ({ as_property: { name, mutations: [{ set: { value } }] } });
  const entries = (name: string, ...items: unknown[][]): any => ({ as_adjacency: { name, mutations: [{ insert: {
    entries: items.map((item) => ({ mutations: item })) } }] } });

  const Meet = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
    (_: any) => _.bob.name("Bob").knows([_.ada]),
    (_: any) => _.ada.name("Ada").age(null).knows([(e: any) => e.known(_.bob).since(2020n)]),
  ]));
  assert(equal(Meet.spec, map([
    { bob: [prop("name", "Bob"), entries("knows", [link("known", ref("ada"))])] },
    { ada: [prop("name", "Ada"), { as_property: { name: "age", mutations: [{ clear: {} }] } },
      entries("knows", [link("known", ref("bob")), prop("since", 2020n)])] },
  ])));
  const book = store.singleton("crm.TheBook");
  const Owned = store.Schemas.OfPattern((p: any) => p.symbols({ bob: Person }).spec( // one callback; a path; a mapping
    (_: any) => _.bob.book({ bob: book }).knows((a: any) => a.insert((e: any) => e.known((n: any) => n.name("Dan")).since(1n)))));
  assert(equal(Owned.spec, map([{ bob: [
    entries("book", [link("book", { ref: pathOf(book).get("ref") }), prop("key", "bob")]),
    entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 1n)]),
  ] }])));
  const Defines = store.Schemas.OfPattern((p: any) => p.symbols({ P: store.Schemas.OfObject }).spec([
    (_: any) => _.P.name("crm.Pet").ref(true).properties({ name: Str, legs: (t: any) => t.schema([Int]) }),
  ]));
  const propertySchema = (schema: unknown): any => ({ def: { schema: "Schemas.OfProperty", mutations: [
    entries("schema", [link("schema", { ref: pathOf(schema).get("ref") })])] } });
  assert(equal(Defines.spec, map([{ P: [prop("name", "crm.Pet"), prop("ref", true), entries(
    "properties", [link("property", propertySchema(Str)), prop("name", "name")],
    [link("property", propertySchema(Int)), prop("name", "legs")])] }])));
  const Terms = store.Schemas.OfPattern((p: any) => p.symbols({ t: store.Mutations.OfTransaction }).spec([
    (_: any) => _.t.uid("u").mutations([(e: any) => e.upsert((u: any) => u.key("k"))]), // a kind union: a kind defined
    (_: any) => _.t.mutations([(e: any) => e.term(_.t)]),
  ]));
  assert(equal(Terms.spec[0], map({ t: [prop("uid", "u"), entries("mutations", [link("term", { def: {
    schema: "Mutations.OfUpsert", mutations: [prop("key", "k")] } })])] })));
  assert(equal(Terms.spec[1], map({ t: [entries("mutations", [link("term", ref("t"))])] })));
  // Reading through the frame, and what a spec refuses.
  const seen: unknown[] = [];
  store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([
    (_: any) => seen.push(repr(_), repr(_.ada), _.ada.name.get(), _.ada.as_schema() === Person,
      _.nope.as_schema(), _.ada.nope.as_schema(), _.ada.knows.as_schema().name,
      "_hidden" in _, "_hidden" in _.ada, "_hidden" in _.ada.name),
    (_: any) => _.ada.knows((a: any) => a.insert((e: any) => (seen.push(
      e.as_schema() === Knows.entry, "_hidden" in e, e.nope.as_schema()), e.known(_.ada)))),
    (_: any) => _.ada.knows([(e: any) => e.known((n: any) => { seen.push(repr(n)); })]),
  ]));
  assert(same(seen, ["<frame>", "<spec of ada>", null, true, null, null, "knows", false, false, false, true, false, null,
    "<spec of an object>"]));
  const Given = store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, bob: Person }).spec([
    (_: any) => _.bob.knows(new Map([[2020n, "Dan"]])).book({ b: _.ada }), // a property of the linked object; an adjacency of it
    (_: any) => _.bob.knows([(e: any) => e.known((n: any) => n.name("Eve")).age(3n)]), // the object it defines, written on
  ]));
  assert(equal(Given.spec[0], map({ bob: [
    entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [prop("name", "Dan")] } }), prop("since", 2020n)]),
    entries("book", [link("book", { def: { schema: "crm.Book", mutations: [entries("contacts", [link("contact", ref("ada"))])] } }),
      prop("key", "b")])] })));
  assert(equal(Given.spec[1], map({ bob: [entries("knows", [link("known", { def: { schema: "crm.Person", mutations: [
    prop("name", "Eve"), prop("age", 3n)] } })])] })));
  const note = store.crm.Note((n: any) => n);
  for (const [spec, error, says] of [
    [(_: any) => _.x.name("x"), TypeError, "symbol 'x' has no schema: a spec writes what its object's schema declares"],
    [(_: any) => _.ada.knows(3n), TypeError, "adjacency 'knows' takes an iterable of entries or of what it links, a mapping"],
    [(_: any) => _.ada.book({ a: 1.5 }), TypeError, "no member of the entry takes 1.5: give a callback"],
    [(_: any) => _.ada.knows([(e: any) => e.known(_.ada).name("Z")]), ValueError,
      "link 'known' links an object a spec does not define: write its members in its own spec"],
    [(_: any) => _.ada.knows([(e: any) => e.known(note)]), ValueError,
      "a counted 'crm.Note' has no path: a spec refers to it through a symbol"],
    [(_: any) => _.ada.meets([(e: any) => e]), TypeError,
      "adjacency 'meets': entries of crm.Meets have more than two links; give each entry's links"],
    [(_: any) => _.ada.trio([(e: any) => e]), TypeError, "adjacency 'trio': entries of its relation have more than two links"],
    [(_: any) => _.ada.knows(new Map([[2020n, _.ada]])), TypeError, "more than one member of the entry takes <spec of ada>: give a callback"],
    [(_: any) => _.ada.meets(new Map([[1n, _.ada]])), TypeError,
      "adjacency 'meets' takes a mapping only where its relation has two links and one property"],
    [(_: any) => _.ada.knows([(e: any) => e.known((n: any) => n.knows([(f: any) => f.known(n)]))]), ValueError,
      "an object a spec defines is linked where it is defined"],
    [(_: any) => _.t.mutations([(e: any) => e.term((x: any) => x)]), TypeError,
      "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
    [(_: any) => _.t.mutations([(e: any) => e.uid("u")]), TypeError,
      "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"],
  ] as [(_: any) => unknown, Function, string][]) {
    raises(error, () => store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person, x: (t: any) => t,
      t: store.Mutations.OfTransaction }).spec([spec])), says);
  }
  const entry: any[] = [];
  store.Schemas.OfPattern((p: any) => p.symbols({ ada: Person }).spec([(_: any) => _.ada.knows([(e: any) => (entry.push(e), e.known(_.ada))])]));
  raises(AttributeError, () => { entry[0].since = 1n; }); // a member is written by calling it
  raises(TypeError, () => new B.PatternBuilder().spec([(_: any) => _]), "a spec is written in a store, whose schemas its symbols name");
  raises(ReadOnlyError, () => new B.PatternBuilder(Meet, false, store).spec([(_: any) => _]));
}

## TXN-05 · A transaction's overlay: data written apart from the store until it commits, dropped where it raises; a schema object that existed changed at its commit

In [ ]:
{
  const store5: any = new B.Store();
  const [Int5, Str5] = [store5.Schemas.OfInt, store5.Schemas.OfStr];
  const [Person5, , Book5, Note5] = crm(store5);
  const book5 = store5.singleton("crm.TheBook");
  let ada5: any = null;
  let note5: any = null;
  let temp5: any = null;
  store5._transacting(() => {
    store5.crm.Book(book5, (b: any) => b.contacts({ ada: (e: any) => e.name("Ada") }));
    ada5 = book5.contacts({ key: "ada" });
    note5 = store5.crm.Note((n: any) => n.text("x"));
    store5.crm.Note(note5, (n: any) => n.text("y"));
    assert(ada5.name.get() === "Ada" && note5.text.get() === "y"); // read through the overlay
  });
  const keys = (): unknown[] => [...book5.contacts].map((e: any) => e.key.get());
  assert(same(keys(), ["ada"]) && note5.text.get() === "y"); // committed
  raises(ValueError, () => store5._transacting(() => {
    store5.crm.Book(book5, (b: any) => b.contacts({ bob: (e: any) => e.name("Bob") }));
    store5.crm.Person(ada5, (c: any) => c.name("Ada L."));
    store5.Schemas.OfObject((o: any) => o.name("crm.Gone").ref(true));
    temp5 = store5.Schemas.OfObject((o: any) => o.ref(true)); // no name of its own, then one
    store5.Schemas.OfObject(temp5, (o: any) => o.name("crm.Temp"));
    throw new ValueError("stop");
  }), "stop");
  assert(same(keys(), ["ada"]) && ada5.name.get() === "Ada"); // dropped
  assert(!store5.names().includes("crm.Gone") && !store5.module("crm").schemas.has("Gone") && !store5.holds(temp5));
  store5._transacting(() => { // a schema object that existed: changed when the transaction commits
    store5.Schemas.OfObject(Note5, (o: any) => o.properties({ tag: Str5 }));
    assert(!Note5.properties.has("tag"));
    store5.Schemas.OfObject(Note5, (o: any) => o.properties({ mood: Str5 })); // from what the one before made
    const Fresh = store5.Schemas.OfObject((o: any) => o.name("crm.Fresh").ref(true));
    store5.Schemas.OfObject(Fresh, (o: any) => o.properties({ n: Int5 })); // one it made: changed at once
    assert(same([...Fresh.properties.keys()], ["n"]));
  });
  assert(same([...Note5.properties.keys()], ["text", "tag", "mood"]));
  raises(ValueError, () => store5._transacting(() => { // refused when it commits: what it wrote is undone, the schemas' and the data's
    store5.Schemas.OfObject(Person5, (o: any) => o.description("people"));
    store5.crm.Note(note5, (n: any) => n.text("z"));
    store5.Schemas.OfObject(Note5, (o: any) => o.name("crm.Book"));
  }), "schema 'crm.Book' is already registered");
  assert(Person5.description === null && Note5.name === "crm.Note" && note5.text.get() === "y");
  assert(store5.registered("crm.Note") === Note5 && store5.registered("crm.Book") === Book5);
  raises(ValueError, () => store5._transacting(() => store5._transacting(() => null)),
    "a transaction is open: transactions do not nest");
}

## TXN-06 · store.transaction: a fixture upserted in one transaction, its frames given back; run again, it finds what it made, an owned object by its spec, a counted one by its frame, key and symbol

In [ ]:
{
  const store6: any = new B.Store();
  const [Person6, , Book6, Note6] = crm(store6);
  const book6 = store6.singleton("crm.TheBook");
  const fixture = (t: any): any => t.uid("fixture").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { ada: Person6, bob: Person6, book: Book6, note: Note6 }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada), bob: (e: any) => e.contact(_.bob) }),
    (_: any) => _.ada.name("Ada").knows([(e: any) => e.known(_.bob).since(2020n)]),
    (_: any) => _.bob.name("Bob").age(null),
    (_: any) => _.note.text("hi").pins(new Map([[1n, _.book]])).odds([(e: any) => e.odd((o: any) => o).k(7n)]), // an Odd defined
  ])]))]);
  const frames = store6.transaction(fixture);
  assert(frames.length === 1 && same([...frames[0].keys()], ["ada", "bob", "book", "note"]) && frames[0].get("book") === book6);
  const [ada6, bob6, note6] = ["ada", "bob", "note"].map((name) => frames[0].get(name));
  assert(equal([...book6.contacts].map((e: any) => [e.key.get(), e.contact.name.get()]), [["ada", "Ada"], ["bob", "Bob"]]));
  assert(equal([...ada6.knows].map((e: any) => [e.known === bob6, e.since.get()]), [[true, 2020n]]));
  assert(equal([...note6.pins].map((e: any) => [e.book === book6, e.at.get()]), [[true, 1n]]) && note6.text.get() === "hi");
  const odd6 = note6.odds[0].odd;
  assert(odd6.as_schema() === store6.registered("crm.Odd") && note6.odds[0].k.get() === 7n);
  const again = store6.transaction(fixture); // found, not made again: nothing to write
  assert(same(["ada", "bob", "note"].map((name) => again[0].get(name)), [ada6, bob6, note6]) && book6.contacts.length === 2);
  assert(ada6.knows.length === 1 && note6.pins.length === 1 && same([...note6.odds].map((e: any) => e.odd), [odd6]));
  const byPath = store6.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person6 }).spec([(_: any) => _.x.book({ ada: book6 })])]))])); // the book by its path
  assert(byPath.length === 1 && byPath[0].size === 1 && byPath[0].get("x") === ada6);
  const built = store6.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.key("other").exists([
    (p: any) => p.symbols({ note: Note6 }).spec([(_: any) => _.note.text("hi")])]))])); // built first, then executed
  const other = store6.transaction(built)[0].get("note");
  assert(other !== note6 && store6.transaction(built)[0].get("note") === other); // a key of its own: another note, kept
  const none = store6.transaction((t: any) => t);
  assert(none.length === 1 && none[0].size === 0); // no term: the one empty frame
  const singleton = store6.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ book: Book6 })]))]));
  assert(singleton.length === 1 && singleton[0].get("book") === book6); // a singleton's symbol binds it
}

## TXN-07 · Map and update: a map binds each object its spec matches, one frame each, the frames before it multiplied; an update writes its spec over each frame

In [ ]:
const store7: any = new B.Store();
const [Person7, , Book7, Note7] = crm(store7);
const book7 = store7.singleton("crm.TheBook");
store7.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
  { a: Person7, b: Person7, c: Person7, book: Book7 }).spec([
  (_: any) => _.book.contacts({ a: (e: any) => e.contact(_.a), b: (e: any) => e.contact(_.b), c: (e: any) => e.contact(_.c) }),
  (_: any) => _.a.name("Ada").age(36n).knows([(e: any) => e.known(_.b).since(2020n), (e: any) => e.known(_.c).since(2021n)]),
  (_: any) => _.b.name("Bob").age(36n).knows([_.c]),
  (_: any) => _.c.name("Cy"),
])]))]));

function mapped(specs: unknown[], symbols: any = null, then: unknown = null): any[] {
  const term = (m: any): any => m.symbols([(p: any) => p.symbols(symbols ?? { x: Person7, y: Person7 }).spec(specs)]);
  return store7.transaction((t: any) => t.mutations([(mu: any) => mu.map(
    then === null ? term : (m: any) => term(m).then([then]))]));
}

const names = (frames: any[]): string[][] => frames.map((frame) => [...frame.values()].map((target: any) => target.name.get()));
{
  assert(equal(names(mapped([(_: any) => _.x.knows([_.y])])), [["Ada", "Bob"], ["Ada", "Cy"], ["Bob", "Cy"]]));
  assert(equal(names(mapped([(_: any) => _.x.knows([(e: any) => e.known((k: any) => k.name("Bob").knows([_.y]))])])),
    [["Ada", "Cy"]])); // through one
  assert(equal(names(mapped([(_: any) => _.x.knows([(e: any) => e.known(_.y).since(2021n)])])), [["Ada", "Cy"]]));
  assert(equal(names(mapped([(_: any) => _.x.age(36n)], { x: Person7 })), [["Ada"], ["Bob"]]));
  const about = mapped([(_: any) => _.x.name("Ada")], { x: Person7 }, (w: any) => w.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ x: Person7, n: Note7 }).spec([(_: any) => _.n.text("about Ada")])])));
  assert(equal(about.map((f: any) => [f.get("x").name.get(), f.get("n").text.get()]), [["Ada", "about Ada"]])); // x as the map bound it
  assert(equal(names(mapped([(_: any) => _.x.age(null)], { x: Person7 })), [["Cy"]])); // a property cleared: none held
  assert(equal(names(mapped([(_: any) => _.x.name("Nobody")], { x: Person7 })), []));
  assert(mapped([], { x: Person7, y: Person7 }).length === 9); // no spec: the cross product
  assert(equal(mapped([(_: any) => _.x.book({ b: _.y })], { x: Person7, y: Book7 }).map((f: any) => [f.get("x").name.get(),
    f.get("y") === book7]), [["Bob", true]])); // an entry's property, through a mapping
  store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person7 }).spec([(_: any) => _.x.age(36n)])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols(
    { x: Person7 }).spec([(_: any) => _.x.age(37n)])]))]))]));
  assert(equal([...store7.extent("crm.Person")].map((p: any) => [p.name.get(), store7.member(p, "age")])
    .sort((x: any, y: any) => (x[0] < y[0] ? -1 : 1)), [["Ada", 37n], ["Bob", 37n], ["Cy", null]]));
  store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: Person7, bk: Book7 }).spec([(_: any) => _.x.name("Bob")])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person7, bk: Book7 }).spec([(_: any) => _.x.age(null).book({ robert: _.bk })])]))]))]));
  const bob7 = book7.contacts({ key: "robert" }); // its one owning entry replaced: its key now "robert"
  assert(bob7.name.get() === "Bob" && store7.member(bob7, "age") === null
    && same([...book7.contacts].map((e: any) => e.key.get()), ["a", "c", "robert"]));
}

## TXN-08 · Reduce and partition: a reduce drops symbols, keeping each frame left once; a partition runs its first, last or every branch, or none where it has no branch, and joins what they end with

In [ ]:
{
  const pairs = (m: any): any => m.symbols([(p: any) => p.symbols({ x: Person7, y: Person7 }).spec([(_: any) => _.x.knows([_.y])])]);
  const dropping = (dropped: string[]) => (r: any): any => r.symbols([(c: any) => c.pattern([(p: any) => p.symbols(
    { x: Person7, y: Person7 })]).symbols(dropped)]);
  const run = (then: unknown): any[] => store7.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => pairs(m).then([then]))]));
  assert(equal(names(run((w: any) => w.reduce(dropping(["y"])))), [["Ada"], ["Bob"]])); // three frames, two left
  assert(equal(names(run((w: any) => w.reduce(dropping(["x"])))), [["Cy"], ["Bob"]])); // in the order the store reaches them
  const labelled = (text: string) => (e: any): any => e.branch((b: any) => b.then([(w: any) => w.upsert((u: any) => u.key(text).exists([
    (p: any) => p.symbols({ n: Note7 }).spec([(_: any) => _.n.text(text)])]))]));
  const arms = (p: any): any => p.branches([labelled("one"), labelled("two")]);
  const partitioned = ({ first = false, last = false, branches = arms, then = null }: { first?: boolean, last?: boolean,
    branches?: (p: any) => any, then?: unknown } = {}): any[] => run((w: any) => w.partition((p: any) => {
    const q = branches(p.first(first).last(last));
    return then === null ? q : q.then([then]);
  }));
  const texts = (frames: any[]): string[] => frames.map((frame) => frame.get("n").text.get());
  const times = (text: string, n: number): string[] => Array(n).fill(text);
  assert(same(texts(partitioned()), [...times("one", 3), ...times("two", 3)])); // every branch, each frame through each
  assert(same(texts(partitioned({ first: true })), times("one", 3)) && same(texts(partitioned({ last: true })), times("two", 3)));
  const otherwise = (p: any): any => p.none([(w: any) => w.reduce(dropping(["y"]))]);
  assert(equal(names(partitioned({ branches: otherwise })), [["Ada"], ["Bob"]])); // no branch: its none
  assert(partitioned({ branches: (p: any) => p }).length === 0); // neither: its frames end
  const joined = partitioned({ then: (w: any) => w.reduce(dropping(["y", "n"])) }); // what the branches end with, joined, then on
  assert(equal(names(joined), [["Ada"], ["Bob"]]));
  assert(run((w: any) => w.partition((p: any) => p.branches([(e: any) => e.branch((b: any) => b)]))).length === 3); // a branch with no then: as it is
}

## TXN-09 · What a transaction refuses before it runs (a symbol no term before binds, an expression, a pattern's variables, a schema object's symbol), and a transaction that raises leaving the store as it was

In [ ]:
{
  const store9: any = new B.Store();
  const [Person9, , Book9] = crm(store9);
  const Int9 = store9.Schemas.OfInt;
  const Exprs = store9.Schemas.OfExpression((e: any) => e.name("crm.True").pattern([(p: any) => p]).result([Int9]));
  const person = (p: any): any => p.symbols({ x: Person9 });
  const refused = (term: unknown, says: string): void =>
    raises(ValueError, () => store9.transaction((t: any) => t.mutations([term])), says);
  refused((mu: any) => mu.update((u: any) => u.apply([person])), "symbol 'x' of an update is bound by no term before it");
  refused((mu: any) => mu.reduce((r: any) => r.symbols([(c: any) => c.pattern([person]).symbols(["x"])])),
    "symbol 'x' of a reduce is bound by no term before it");
  refused((mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person9 }).requires([Exprs])])),
    "a map's pattern requires constraints: transactions evaluate no expressions until step 4h");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.variables({ v: Int9 })])), "an upsert's pattern has variables");
  refused((mu: any) => mu.map((m: any) => m.variables([person])), "a map of variables");
  refused((mu: any) => mu.map((m: any) => m.symbols([person]).then([(w: any) => w.reduce((r: any) => r.symbols([
    (c: any) => c.pattern([person]).symbols(["x"]).suffixes({ current: "_a", next: "_b" })]))])), "a reduce that collects");
  refused((mu: any) => mu.partition((p: any) => p.first(true).last(true)),
    "a partition dispatches to its first matching branch or its last, not both");
  refused((mu: any) => mu.partition((p: any) => p.branches([(e: any) => e.branch((b: any) => b.requires([Exprs]))])),
    "a branch requires constraints");
  refused((mu: any) => mu.map((m: any) => m), "a map takes one pattern in 'symbols', not 0");
  refused((mu: any) => mu.reduce((r: any) => r), "a reduce takes one collector in 'symbols', not 0");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ s: store9.Schemas.OfObject })])),
    "symbol 's' binds a schema object");
  refused((mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ s: Int9 })])),
    "symbol 's' binds no reference object of one schema");
  refused((mu: any) => mu.map((m: any) => m.symbols([person]).then([(w: any) => w.partition((p: any) => p.branches([
    (e: any) => e.branch((b: any) => b.then([(w2: any) => w2.upsert((u: any) => u.exists([(q: any) => q.symbols({ y: Person9 })]))])),
    (e: any) => e.branch((b: any) => b),
  ]).then([(w2: any) => w2.update((u: any) => u.apply([(q: any) => q.symbols({ y: Person9 })]))]))])),
  "symbol 'y' of an update is bound by no term before it"); // bound in one branch only
  const book9 = store9.singleton("crm.TheBook");
  const ada9 = store9.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { ada: Person9, book: Book9 }).spec([(_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada) }),
    (_: any) => _.ada.name("Ada")])]))]))[0].get("ada");
  const orphaning = (t: any): any => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person9 })
    .spec([(_: any) => _.x.name("Ada")])]).then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols({ x: Person9 })
    .spec([(_: any) => _.x.age(1n).knows([(e: any) => e.known(_.x)])])]).then([(w2: any) => w2.upsert((u2: any) => u2.exists([
    (p: any) => p.symbols({ o: Person9 }).spec([(_: any) => _.o.name("Orphan")])]))]))]))]);
  raises(ValueError, () => store9.transaction(orphaning), "a 'crm.Person' needs an owner"); // refused when the transaction ends
  assert(store9.member(ada9, "age") === null && ada9.knows.length === 0 && same([...store9.extent("crm.Person")], [ada9]));
  store9.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { bea: Person9, book: Book9 }).spec([(_: any) => _.book.contacts({ bea: (e: any) => e.contact(_.bea.name("Bea")) })])]))]));
  const knowing = store9.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: Person9 }).spec([(_: any) => _.x.name("Bea")])]).then([(w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person9 }).spec([(_: any) => _.x.knows([ada9])])]))]))])); // Ada by her path
  store9.crm.Book(book9, (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("ada")))); // Ada deleted
  raises(LookupError, () => store9.transaction(knowing), "no object of the store has the path a spec refers to");
  assert(same([...book9.contacts].map((e: any) => e.contact.name.get()), ["Bea"]));
}